In [2]:
import pandas as pd

# Path provided in your snippet
data_path = "benchmark_merged_labeled_data_noisy.csv"
output_path = "benchmark_balanced_noisy.csv"

try:
    df = pd.read_csv(data_path)
    
    print("--- Original Distribution ---")
    print(df['label'].value_counts())

    # 1. Determine the target count based on WALKING
    # We use WALKING as the gold standard for how much data a "majority" class should have
    if 'WALKING' in df['label'].values:
        target_count = df[df['label'] == 'WALKING'].shape[0]
    else:
        # Fallback to a reasonable number if WALKING is missing
        target_count = 350 
        
    print(f"\nTarget cap for STANDING: {target_count} samples")

    # 2. Separate the 'bully' class
    df_standing = df[df['label'] == 'STANDING']
    df_others = df[df['label'] != 'STANDING']

    # 3. Downsample STANDING to match WALKING
    if len(df_standing) > target_count:
        df_standing_balanced = df_standing.sample(n=target_count, random_state=42)
        print(f"Reduced STANDING from {len(df_standing)} to {target_count}")
    else:
        df_standing_balanced = df_standing
        print("STANDING count is already below target.")

    # 4. Recombine
    # We keep SITTING, UPSTAIRS, and DOWNSTAIRS exactly as they are
    balanced_df = pd.concat([df_standing_balanced, df_others]).reset_index(drop=True)

    print("\n--- New Balanced Distribution ---")
    print(balanced_df['label'].value_counts())

    # 5. Save the new benchmark
    balanced_df.to_csv(output_path, index=False)
    print(f"\nSuccess! Balanced benchmark saved to: {output_path}")

except FileNotFoundError:
    print(f"Error: Could not find {data_path}. Ensure the file is in the correct directory.")

--- Original Distribution ---
label
STANDING              83308
WALKING               33353
WALKING_DOWNSTAIRS    10917
SITTING                7433
WALKING_UPSTAIRS       6838
LAYING                 2846
Name: count, dtype: int64

Target cap for STANDING: 33353 samples
Reduced STANDING from 83308 to 33353

--- New Balanced Distribution ---
label
STANDING              33353
WALKING               33353
WALKING_DOWNSTAIRS    10917
SITTING                7433
WALKING_UPSTAIRS       6838
LAYING                 2846
Name: count, dtype: int64

Success! Balanced benchmark saved to: benchmark_balanced_noisy.csv
